In [0]:
from pyspark.sql.functions import *
import re


df = spark.table("bronze.encounters")


# fixing column and making it in snakecase

df = df.toDF(*[
    re.sub(r'[^a-z0-9_]', '', c.lower().replace(" ", "_"))
    for c in df.columns
])

# handling unknown and null and blank spaces

df = df.replace(["", "unknown", "UNKNOWN"], None)

# 4. fixing timestamp format (ISO format)

df = df.withColumn(
    "start",
    to_timestamp(col("start"), "yyyy-MM-dd'T'HH:mm:ss.SSSXXX")
).withColumn(
    "stop",
    to_timestamp(col("stop"), "yyyy-MM-dd'T'HH:mm:ss.SSSXXX")
)

# numeric columns(handle commas + scientific notation)


# remove commas
df = df.withColumn("base_encounter_cost", regexp_replace(col("base_encounter_cost"), ",", "")) \
       .withColumn("total_claim_cost", regexp_replace(col("total_claim_cost"), ",", "")) \
       .withColumn("payer_coverage", regexp_replace(col("payer_coverage"), ",", "")) \
       .withColumn("code", regexp_replace(col("code"), ",", "")) \
       .withColumn("reasoncode", regexp_replace(col("reasoncode"), ",", ""))

# cast properly
df = df.withColumn("base_encounter_cost", col("base_encounter_cost").cast("double")) \
       .withColumn("total_claim_cost", col("total_claim_cost").cast("double")) \
       .withColumn("payer_coverage", col("payer_coverage").cast("double")) \
       .withColumn("code", col("code").cast("double").cast("long")) \
       .withColumn("reasoncode", col("reasoncode").cast("double").cast("long"))

# 6. CLEAN STRING COLUMNS

df = df.withColumn("description", trim(col("description"))) \
       .withColumn("reasondescription", trim(col("reasondescription"))) \
       .withColumn("encounter_class", lower(trim(col("encounter_class"))))


# 7. DATA QUALITY RULES

# remove invalid records (critical fields)
df = df.filter(
    col("patient").isNotNull() &
    col("id").isNotNull()
)

# ensure cost is not negative
df = df.filter(
    (col("base_encounter_cost").isNull() | (col("base_encounter_cost") >= 0)) &
    (col("total_claim_cost").isNull() | (col("total_claim_cost") >= 0))
)


# Remove system columns (like _line, _fivetran_synced)
df = df.select([
    col(c) for c in df.columns if not c.startswith("_")
])


# Save to silver
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.encounters")